# Phase 1.5 — Natal Embedding Visualization

Extract `natal_emb` (256-d) from a trained NatalEncoder for 1k–5k charts, reduce to 2D with t-SNE or UMAP, and color by:
- **Dominant element** (argmax of elemental_strengths)
- **Useful god** (用神, 6-class: 0–4 + 5=ambiguous)
- **Day-master strength** (binned: weak / medium / strong)

Cluster quality is documented in the notes at the end.

In [1]:
# Add project root for imports (run from repo root or from notebooks/)
import sys
from pathlib import Path

_root = Path.cwd() if (Path.cwd() / "bazi_engine").is_dir() else Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import json
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Config: paths and evaluation settings
DATA_PATH = _root / "data" / "charts.jsonl"
CHECKPOINT_PATH = _root / "checkpoints" / "ten_god_best.pt"  # or multi_task_best.pt
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
VAL_RATIO = 0.15
BATCH_SIZE = 64
MAX_SAMPLES = 3000  # cap charts for faster t-SNE/UMAP (1k–5k per plan)

In [ ]:
# Imports for encoder, dataset, and model construction (for checkpoint loading)
from bazi_model.data import ChartDataset, chart_collate_fn_inputs_targets
from bazi_model.encoders.natal_encoder import NatalEncoder
from bazi_model.tasks.ten_god_head import TenGodHead
from bazi_engine.models import BaziChart
from bazi_engine.strength import compute_day_master_strength

# NatalEncoder.forward expects these input keys (no ground truth)
NATAL_INPUT_KEYS = (
    "stem_ids", "branch_ids", "stem_elements", "stem_polarities", "pillar_positions",
)

In [ ]:
# Load NatalEncoder from checkpoint (single-task checkpoint: encoder + head)
# Build same wrapper as training so state_dict keys match, then use encoder only.
class _TenGodWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = NatalEncoder()
        self.head = TenGodHead(d_model=256)

    def forward(self, **kwargs):
        return self.head(self.encoder(**kwargs))

wrapper = _TenGodWrapper().to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    wrapper.load_state_dict(ckpt["model_state_dict"], strict=True)
else:
    wrapper.load_state_dict(ckpt, strict=True)
encoder = wrapper.encoder
encoder.eval()
print(f"Loaded encoder from {CHECKPOINT_PATH.name}")

In [ ]:
# Dataset and val split (natal-only, same as training)
torch.manual_seed(SEED)
full_ds = ChartDataset(DATA_PATH, task_name="multi", temporal_only=False)
n_val = max(1, int(len(full_ds) * VAL_RATIO))
n_train = len(full_ds) - n_val
_, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=chart_collate_fn_inputs_targets,
    num_workers=0,
)
print(f"Val samples: {len(val_ds)}")

In [ ]:
# Extract natal_emb and labels (elemental_strengths, useful_god_id)
emb_list = []
el_strengths_list = []
ug_id_list = []

with torch.no_grad():
    for inputs, targets in val_loader:
        if len(emb_list) * BATCH_SIZE >= MAX_SAMPLES:
            break
        encoder_kw = {k: inputs[k].to(DEVICE) for k in NATAL_INPUT_KEYS if k in inputs}
        natal_emb = encoder(**encoder_kw)
        emb_list.append(natal_emb.cpu().numpy())
        el_strengths_list.append(targets["elemental_strengths"].numpy())
        ug_id_list.append(targets["useful_god_id"].numpy().squeeze(-1))

embeddings = np.concatenate(emb_list, axis=0)[:MAX_SAMPLES]
elemental_strengths = np.concatenate(el_strengths_list, axis=0)[:MAX_SAMPLES]
useful_god_id = np.concatenate(ug_id_list, axis=0)[:MAX_SAMPLES]
print(f"Collected embeddings: {embeddings.shape}, elemental_strengths: {elemental_strengths.shape}, useful_god_id: {useful_god_id.shape}")

In [ ]:
# Compute day_master_strength for each val sample (chart not in targets; parse from file)
n_used = len(embeddings)
val_indices = val_ds.indices  # global indices into full dataset
with open(DATA_PATH, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]
day_master_strengths = []
for idx in val_indices[:n_used]:
    chart = BaziChart.from_json(lines[idx])
    stems, branches = chart._stems_and_branches()
    dm = compute_day_master_strength(stems, branches)
    day_master_strengths.append(dm)
day_master_strength = np.array(day_master_strengths, dtype=np.float64)
print(f"day_master_strength: min={day_master_strength.min():.3f}, max={day_master_strength.max():.3f}")

In [ ]:
# Dimensionality reduction: t-SNE and optionally UMAP
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000)
coords_tsne = tsne.fit_transform(embeddings)
print("t-SNE done. Shape:", coords_tsne.shape)

In [ ]:
# Optional: UMAP (install with pip install umap-learn)
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED)
    coords_umap = reducer.fit_transform(embeddings)
    use_umap = True
    print("UMAP done. Shape:", coords_umap.shape)
except ImportError:
    use_umap = False
    coords_umap = None
    print("UMAP skipped (umap-learn not installed). Using t-SNE only.")

In [ ]:
# Labels for plotting
ELEMENT_LABELS = ("Wood 木", "Fire 火", "Earth 土", "Metal 金", "Water 水")
dominant_element = np.argmax(elemental_strengths, axis=1)  # (n_used,)

# Bin day_master_strength into weak / medium / strong (tertiles)
q1, q2 = np.percentile(day_master_strength, [33.33, 66.67])
strength_bin = np.zeros(n_used, dtype=int)
strength_bin[day_master_strength <= q1] = 0
strength_bin[(day_master_strength > q1) & (day_master_strength <= q2)] = 1
strength_bin[day_master_strength > q2] = 2
STRENGTH_LABELS = ("Weak", "Medium", "Strong")

In [ ]:
# Scatter: t-SNE colored by dominant element
fig, ax = plt.subplots(figsize=(10, 8))
for i, label in enumerate(ELEMENT_LABELS):
    mask = dominant_element == i
    ax.scatter(coords_tsne[mask, 0], coords_tsne[mask, 1], label=label, alpha=0.6, s=8)
ax.set_title("Natal embeddings (t-SNE) — colored by dominant element")
ax.legend(loc="best", fontsize=8)
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: t-SNE colored by useful god (0–4 = 用神 type, 5 = ambiguous)
fig, ax = plt.subplots(figsize=(10, 8))
ug_labels = ["比肩", "劫财", "食神", "伤官", "偏财", "Ambiguous"]
for i in range(6):
    mask = useful_god_id == i
    if mask.any():
        ax.scatter(coords_tsne[mask, 0], coords_tsne[mask, 1], label=ug_labels[i], alpha=0.6, s=8)
ax.set_title("Natal embeddings (t-SNE) — colored by useful god (用神)")
ax.legend(loc="best", fontsize=8)
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: t-SNE colored by day-master strength (weak / medium / strong)
fig, ax = plt.subplots(figsize=(10, 8))
for i, label in enumerate(STRENGTH_LABELS):
    mask = strength_bin == i
    ax.scatter(coords_tsne[mask, 0], coords_tsne[mask, 1], label=label, alpha=0.6, s=8)
ax.set_title("Natal embeddings (t-SNE) — colored by day-master strength")
ax.legend(loc="best", fontsize=8)
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

In [ ]:
# Optional: same three colorings on UMAP
if use_umap and coords_umap is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    # Dominant element
    for i, label in enumerate(ELEMENT_LABELS):
        mask = dominant_element == i
        axes[0].scatter(coords_umap[mask, 0], coords_umap[mask, 1], label=label, alpha=0.6, s=8)
    axes[0].set_title("Dominant element"); axes[0].legend(loc="best", fontsize=6)
    # Useful god
    for i in range(6):
        mask = useful_god_id == i
        if mask.any():
            axes[1].scatter(coords_umap[mask, 0], coords_umap[mask, 1], label=ug_labels[i], alpha=0.6, s=8)
    axes[1].set_title("Useful god (用神)"); axes[1].legend(loc="best", fontsize=6)
    # Day-master strength
    for i, label in enumerate(STRENGTH_LABELS):
        mask = strength_bin == i
        axes[2].scatter(coords_umap[mask, 0], coords_umap[mask, 1], label=label, alpha=0.6, s=8)
    axes[2].set_title("Day-master strength"); axes[2].legend(loc="best", fontsize=6)
    plt.suptitle("Natal embeddings (UMAP)")
    plt.tight_layout()
    plt.show()

---

## Cluster quality notes (completion criteria)

- **Dominant element:** Inspect whether the five elements (Wood, Fire, Earth, Metal, Water) form visually distinct regions in t-SNE/UMAP. Some overlap is expected (e.g. 五行 relationships); note if one element is heavily mixed with others or if any element clusters clearly.
- **Useful god (用神):** Check if the 6 classes (5 types + ambiguous) show separation. Ambiguous (class 5) may spread across the space; unambiguous types may form looser clusters. Document whether clustering is "meaningful" (same-label points tend to be nearby).
- **Day-master strength:** Weak / medium / strong bins should show some gradient or separation if the encoder captures strength-related structure; strong overlap suggests the representation is not strongly aligned with day-master strength in 2D.
- **Caveats:** (1) t-SNE/UMAP preserve local structure; global distances are not interpretable. (2) Subset size (e.g. 1k–3k) and perplexity/n_neighbors affect layout. (3) No formal metric is required; qualitative "meaningful clustering" per completion criteria suffices.